# Olist E-Commerce: SQL Analysis
Business questions answered with SQL on the Olist Brazilian e-commerce dataset.


### Importing required libraries

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

### path of the data

In [2]:
path = Path(r'C:\Users\EXPRESS\Desktop\E-Commerce-Customer-Value-and-Purchase-Prediction--main\data\processed\parquet')

for file in path.iterdir():
    print(file.name)

olist_customers_dataset.parquet
olist_geolocation_dataset.parquet
olist_orders_dataset.parquet
olist_order_items_dataset.parquet
olist_order_payments_dataset.parquet
olist_order_reviews_dataset.parquet
olist_products_dataset.parquet
olist_sellers_dataset.parquet
product_category_name_translation.parquet


In [3]:
customers = pd.read_parquet(f'{path}/olist_customers_dataset.parquet')
geolocation = pd.read_parquet(f'{path}/olist_geolocation_dataset.parquet')
orders = pd.read_parquet(f'{path}/olist_orders_dataset.parquet')
order_items = pd.read_parquet(f'{path}/olist_order_items_dataset.parquet')
payments = pd.read_parquet(f'{path}/olist_order_payments_dataset.parquet')
reviews = pd.read_parquet(f'{path}/olist_order_reviews_dataset.parquet')
products = pd.read_parquet(f'{path}/olist_products_dataset.parquet')
sellers = pd.read_parquet(f'{path}/olist_sellers_dataset.parquet')
category_translation = pd.read_parquet(f'{path}/product_category_name_translation.parquet')

In [4]:
print("customers:", customers.shape)
print("geolocation:", geolocation.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("category_translation:", category_translation.shape)

customers: (99441, 5)
geolocation: (738332, 5)
orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 5)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


### Database Connection

In [5]:
db = sqlite3.connect('olist_analysis.db')

In [6]:
customers.to_sql('customers', db, if_exists='replace', index=False)
geolocation.to_sql('geolocation', db, if_exists='replace', index=False)
orders.to_sql('orders', db, if_exists='replace', index=False)
order_items.to_sql('order_items', db, if_exists='replace', index=False)
payments.to_sql('payments', db, if_exists='replace', index=False)
reviews.to_sql('reviews', db, if_exists='replace', index=False)
products.to_sql('products', db, if_exists='replace', index=False)
sellers.to_sql('sellers', db, if_exists='replace', index=False)
category_translation.to_sql('category_translation', db, if_exists='replace', index=False)

71

In [7]:
def run(query):
    return pd.read_sql_query(query, db)

## 1. Sales and business performance

### Q1. How has monthly sales revenue changed over time?
**Business purpose:** Spot high- and low-performing months.

In [24]:
run("""
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
       ROUND(SUM(oi.price), 2) AS total_revenue,
       COUNT(DISTINCT o.order_id) AS total_orders
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
  AND month BETWEEN '2017-01' AND '2018-08'
GROUP BY month
ORDER BY total_revenue DESC
""")

,month,total_revenue,total_orders
0,2017-11,987765.37,7289
1,2018-05,977544.69,6749
2,2018-04,973534.09,6798
3,2018-03,953356.25,7003
4,2018-01,924645.00,7069
5,2018-07,867953.46,6159
6,2018-06,856077.86,6099
7,2018-08,838576.64,6351
8,2018-02,826437.13,6555
9,2017-12,726033.19,5513


**Takeaway:** Revenue grows strongly through 2017 and peaks in **Nov 2017** (~988K, ~7.3K orders), likely Black Friday. It stays high through 2018.

### Q2. Which product categories generate the highest revenue?
**Business purpose:** Find the biggest categories by sales value.

In [26]:
run("""
SELECT ct.product_category_name_english AS category,
       ROUND(SUM(oi.price), 2) AS total_revenue,
       COUNT(DISTINCT oi.order_id) AS total_orders
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
LEFT JOIN category_translation ct ON ct.product_category_name = p.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10
""")

,category,total_revenue,total_orders
0,health_beauty,1233131.72,8647
1,watches_gifts,1166176.98,5495
2,bed_bath_table,1023434.76,9272
3,sports_leisure,954852.55,7530
4,computers_accessories,888724.61,6530
5,furniture_decor,711927.69,6307
6,housewares,615628.69,5743
7,cool_stuff,610204.10,3559
8,auto,578966.65,3810
9,toys,471286.48,3804


**Takeaway:** Health & beauty, watches & gifts and bed/bath/table lead in revenue. Revenue is spread across many categories, not one dominant one.

### Q3. What is the average order value (AOV) over time?
**Business purpose:** See whether customers spend more or less per order.

In [10]:
run("""
SELECT month, ROUND(AVG(order_total), 2) AS avg_order_value
FROM (
    SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
           o.order_id,
           SUM(oi.price) AS order_total
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND strftime('%Y-%m', o.order_purchase_timestamp) BETWEEN '2017-01' AND '2018-08'
    GROUP BY o.order_id
)
GROUP BY month
ORDER BY month
""")

,month,avg_order_value
0,2017-01,149.06
1,2017-02,141.70
2,2017-03,141.08
3,2017-04,147.92
4,2017-05,138.00
5,2017-06,134.58
6,2017-07,124.38
7,2017-08,132.29
8,2017-09,146.36
9,2017-10,144.76


**Takeaway:** AOV is stable at roughly 125–150 per month. Revenue growth came from **more orders**, not bigger baskets.

## 2. Customer behavior

### Q4. How many customers bought more than once vs. only once?
**Business purpose:** Measure retention and repeat purchasing.

In [11]:
run("""
SELECT COUNT(*) AS unique_customers,
       SUM(order_count > 1) AS repeat_customers,
       ROUND(SUM(order_count > 1) * 100.0 / COUNT(*), 2) AS repeat_percentage
FROM (
    SELECT c.customer_unique_id, COUNT(o.order_id) AS order_count
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
""")

,unique_customers,repeat_customers,repeat_percentage
0,96096,2997,3.12


**Takeaway:** Only about **3%** of customers buy again. Retention is weak, which is the biggest growth opportunity (loyalty offers, follow-up emails).

### Q5. Which payment methods are used most, and how do they differ in value?
**Business purpose:** Understand payment preferences.

In [12]:
run("""
SELECT payment_type,
       COUNT(*) AS payments,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM payments WHERE payment_type <> 'not_defined'), 1) AS pct_of_payments,
       ROUND(AVG(payment_value), 2) AS avg_payment_value,
       ROUND(SUM(payment_value), 2) AS total_payment_value
FROM payments
WHERE payment_type <> 'not_defined'
GROUP BY payment_type
ORDER BY payments DESC
""")

,payment_type,payments,pct_of_payments,avg_payment_value,total_payment_value
0,credit_card,76795,73.9,163.32,12542084.19
1,boleto,19784,19.0,145.03,2869361.27
2,voucher,5775,5.6,65.70,379436.87
3,debit_card,1529,1.5,142.57,217989.79


**Takeaway:** Credit card dominates (about 73.9% of payment value), followed by boleto. 

### Q6. What percentage of orders arrive later than the estimated date?
**Business purpose:** Check whether delivery promises are being met.

In [13]:
run("""
SELECT COUNT(*) AS delivered_orders,
       SUM(order_delivered_customer_date > order_estimated_delivery_date) AS late_orders,
       ROUND(SUM(order_delivered_customer_date > order_estimated_delivery_date) * 100.0 / COUNT(*), 2) AS late_percentage
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
""")

,delivered_orders,late_orders,late_percentage
0,96470,7826,8.11


**Takeaway:** About **8%** of orders are late. Most orders meet the promise, but late deliveries are a likely driver of bad reviews.

### Q7. What is the review score distribution?
**Business purpose:** Measure customer satisfaction.

In [14]:
run("""
SELECT review_score,
       COUNT(*) AS total_reviews,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM reviews), 1) AS pct_of_reviews
FROM reviews
GROUP BY review_score
ORDER BY review_score DESC
""")

,review_score,total_reviews,pct_of_reviews
0,5,57328,57.8
1,4,19142,19.3
2,3,8179,8.2
3,2,3151,3.2
4,1,11424,11.5


**Takeaway:** About 77% of reviews are 4–5 stars, but roughly **15%** are 1–2 stars. 

## 3. Sellers, regions and logistics

### Q8. Who are the top 10 sellers by revenue?
**Business purpose:** Identify the sellers that matter most to the marketplace.

In [15]:
run("""
SELECT s.seller_id, s.seller_state,
       ROUND(SUM(oi.price), 2) AS total_revenue,
       COUNT(DISTINCT oi.order_id) AS total_orders
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN sellers s ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_id, s.seller_state
ORDER BY total_revenue DESC
LIMIT 10
""")

,seller_id,seller_state,total_revenue,total_orders
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,226987.93,1124
1,53243585a1d6dc2643021fd1853d8905,BA,217940.44,348
2,4a3ca9315b744ce9f8e9374361493884,SP,196882.12,1772
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,190917.14,578
4,7c67e1448b00f6e969d365cea6b010ab,SP,186570.05,973
5,7e93a43ef30c4f03f38b393420bc753a,SP,165981.49,319
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,159816.87,1311
7,7a67c85e85bb2ce8582c35f2203ad736,SP,139658.69,1145
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,138208.56,910
9,955fee9216a65b617aa5c0531780ce60,SP,131836.71,1261


**Takeaway:** A small group of sellers brings in a large share of sales. Losing one of them would be noticeable.

### Q9. Which states place the most orders?
**Business purpose:** Find the strongest regional markets.

In [16]:
run("""
SELECT c.customer_state,
       COUNT(DISTINCT o.order_id) AS total_orders,
       ROUND(COUNT(DISTINCT o.order_id) * 100.0 / (SELECT COUNT(*) FROM orders), 1) AS pct_of_orders
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_orders DESC
LIMIT 10
""")

,customer_state,total_orders,pct_of_orders
0,SP,41746,42.0
1,RJ,12852,12.9
2,MG,11635,11.7
3,RS,5466,5.5
4,PR,5045,5.1
5,SC,3637,3.7
6,BA,3380,3.4
7,DF,2140,2.2
8,ES,2033,2.0
9,GO,2020,2.0


**Takeaway:** **São Paulo (SP)** alone has about 42% of all orders, followed by RJ and MG. Demand is heavily concentrated in.

### Q10. Which categories have the highest average freight cost?
**Business purpose:** Find where shipping costs are highest.

In [ ]:
run("""
SELECT ct.product_category_name_english AS category,
       ROUND(AVG(oi.freight_value), 2) AS avg_freight_cost
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY category
ORDER BY avg_freight_cost DESC
LIMIT 10
""")

,category,avg_freight_cost
0,computers,48.45
1,home_appliances_2,44.54
2,furniture_mattress_and_upholstery,42.91
3,kitchen_dining_laundry_garden_furniture,42.70
4,furniture_bedroom,42.50
5,office_furniture,40.55
6,small_appliances_home_oven_and_coffee,36.16
7,furniture_living_room,35.72
8,signaling_and_security,32.70
9,industry_commerce_and_business,29.42


**Takeaway:** Bulky items (computers, appliances, furniture) have the highest freight, around 42–48 per item, while light items like clothes and books are near 12–13.

## Summary
- Sales grew fast through 2017 and 2018, driven by **order volume**, not bigger orders.
- **Retention is weak**: only ~3% of customers buy again.
- Delivery is mostly on time (~92%), but ~15% of reviews are negative.
- Demand is concentrated in **São Paulo** and a few categories lead revenue.
- Heavy items cost the most to ship.